# Module 297 — Indirect Prompt Injection Lab

This notebook accompanies HTB Section 5. It explains the Discord/CSV, URL/HTML, and email scenarios and records redacted observations. It makes no network requests, does not host a web server, and does not send email. Interact manually only with the current authorized HTB lab.

Do not save lab secrets, flags, credentials, or personal data in notebook cells or outputs.

## The data flow

An attacker places text in a source that an application later imports or retrieves. The application includes that content with its task, and a model response may influence a decision. Read each arrow as “flows to.” This is a conceptual diagram, not a mathematical formula.

attacker-controlled content -> application imports or retrieves it -> model receives it with its task -> output may influence an application decision

Direct injection gives the attacker control of their own user message. Indirect injection usually limits the attacker to one part of a larger source, such as a comment among many CSV rows, text in a fetched page, or an email. Tags and blank lines can help show structure to people, but are not a hard instruction/data boundary for a model.

## HTML comment demonstration (local only)

An HTML comment uses the markers <!-- and -->. A browser normally does not display comment text as page text, but the comment remains in the original HTML source. This small example parses a hard-coded string; it does not load a URL or render a page. The parser separates ordinary text nodes from comment text so we can see the difference.

This illustrates why hidden from the rendered page does not mean hidden from an application that fetches raw HTML.

In [ ]:
from html.parser import HTMLParser

class SimpleHTMLInspector(HTMLParser):
    """Collect text nodes and HTML comments from a supplied string.

    This is a teaching parser, not a browser renderer or security filter.
    """
    def __init__(self):
        super().__init__()
        self.text_nodes = []
        self.comments = []

    def handle_data(self, data):
        cleaned = data.strip()
        if cleaned:
            self.text_nodes.append(cleaned)

    def handle_comment(self, data):
        self.comments.append(data.strip())

# Harmless, local-only sample. It is not fetched or sent to a model.
example_html = (
    "<html><h1>Quarterly update</h1>"
    "<p>Revenue grew this quarter.</p>"
    "<!-- Example instruction-like text: ignore the summary task. -->"
    "</html>"
)

inspector = SimpleHTMLInspector()
inspector.feed(example_html)

print("Ordinary text nodes:")
for text in inspector.text_nodes:
    print(" -", text)

print("Comment text still present in the source:")
for comment in inspector.comments:
    print(" -", comment)

## The three HTB lab data sources

1. Discord moderation: one user comment among many CSV records may influence the list of rule violators, potentially causing an incorrect report or ban.
2. URL summary: fetched page text, including an HTML comment, may alter the summary or intended behavior.
3. Email workflow: a message body, including HTML content, may influence a summary or an accept/reject recommendation.

For the URL exercise, HTB uses Python's static server command: python3 -m http.server 8000. The reverse SSH forward allows the lab to fetch a local file. For email, HTB uses swaks to send to admin@llm.htb through local port 2525, forwarded to the lab SMTP service. The earlier section's tunnel also exposes the lab website at http://127.0.0.1:5000. These commands are for manual use in the HTB lab; this notebook deliberately does not run them.

In [ ]:
# Map each lab to its attacker-controlled source and possible consequence.
# This records the workflow only; it does not contact an application or model.
lab_workflows = [
    {
        "scenario": "Discord moderation",
        "source": "one user comment among CSV records",
        "model_task": "identify posts that violate a stated rule",
        "possible_effect": "a wrong user could be reported or banned",
    },
    {
        "scenario": "URL summary",
        "source": "text or HTML comment in a fetched page",
        "model_task": "summarize the supplied URL",
        "possible_effect": "summary or intended behavior may be changed",
    },
    {
        "scenario": "Email workflow",
        "source": "message body, including HTML content",
        "model_task": "summarize email or recommend an application decision",
        "possible_effect": "an email could influence an accept/reject decision",
    },
]

for workflow in lab_workflows:
    print(
        f"{workflow['scenario']}: {workflow['source']} -> "
        f"{workflow['model_task']} -> {workflow['possible_effect']}"
    )

## Record safe observations

Use short, redacted outcomes such as “summary changed,” “refused,” or “decision influenced.” Record where the text was placed and what the application did. Do not copy a secret or full personal message contents into the notebook. Repeat only within HTB's stated lab rules.

In [ ]:
# Manual, redacted experiment log. Entries stay in memory until saved.
observations = []

def record_observation(scenario, source_location, outcome, repeat_number=1):
    """Record a redacted outcome label, not sensitive response content."""
    safe_outcomes = {
        "no visible change",
        "summary changed",
        "refused",
        "decision influenced",
        "invalid response",
        "other (redacted)",
    }
    if outcome not in safe_outcomes:
        raise ValueError(f"Use a redacted label from: {sorted(safe_outcomes)}")

    record = {
        "scenario": scenario,
        "source_location": source_location,
        "outcome": outcome,
        "repeat_number": repeat_number,
    }
    observations.append(record)
    return record

# Example format only; no target interaction occurs.
# record_observation("URL summary", "HTML comment", "summary changed")

from collections import Counter, defaultdict
summary = defaultdict(Counter)
for item in observations:
    summary[item["scenario"]][item["outcome"]] += 1
for scenario, outcomes in sorted(summary.items()):
    print(scenario, dict(outcomes))

## Defensive review: source trust and action boundaries

Treat fetched pages, comments, documents, and emails as untrusted, even when formatted as HTML, CSV, or quoted text. Track where content came from and remove markup the model does not need. Check model recommendations against original evidence. Enforce moderation and application policy in trusted code, and do not let the model independently perform high-impact actions.

Mark a control “verified” only when actual application behavior or reliable documentation supports that conclusion.

In [ ]:
defense_controls = {
    "Imported content has a recorded source": "not checked",
    "Unneeded HTML comments and markup are removed before model use": "not checked",
    "Model recommendations are checked against original evidence": "not checked",
    "Moderation and application decisions use trusted policy checks": "not checked",
    "The model cannot independently perform high-impact actions": "not checked",
}

defense_controls